In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Citirea datelor

In [2]:
url = '3.input_data_prepped_bow.csv'
reviews = pd.read_csv(url)
reviews.head(2)

,rest_id,text,rating,char_count,positive,text_prep,text_prep_tokens,word_len_prep,text_prep_lim,text_prep_lim_tokens,word_len_prep_lim
0,yGMCl0vYigshkXiZFIDTNw,We arrived for our reservation at 7:15pm. The...,4,302,1,arrived reservation pm seat -PRON- right time ...,"['arrived', 'reservation', 'pm', 'seat', '-PRO...",27,arrived reservation pm seat right time restura...,"['arrived', 'reservation', 'pm', 'seat', 'righ...",25
1,yGMCl0vYigshkXiZFIDTNw,We received amazing service again. The food wa...,5,111,1,receive amazing service food cook right waitre...,"['receive', 'amazing', 'service', 'food', 'coo...",10,receive amazing service food cook right waitre...,"['receive', 'amazing', 'service', 'food', 'coo...",9


In [3]:
reviews.shape

(9365, 11)

In [4]:
url = 'dtm_1_bow.parquet'
dtm_bow = pd.read_parquet(url)

In [5]:
dtm_bow.shape

(9365, 6000)

# Train test split

In [6]:
X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(
    dtm_bow,
    reviews['positive'],
    train_size=0.8,
    random_state=42
    )

In [7]:
print(len(X_train_bow), len(X_test_bow), len(y_train_bow), len(y_test_bow))

7492 1873 7492 1873


# Model

In [8]:
#initializarea obiectului
classifier_rf = RandomForestClassifier(random_state=42, n_jobs=10, max_depth=5,
                                       n_estimators=100, oob_score=True)


In [9]:
#training
classifier_rf.fit(X_train_bow, y_train_bow)

RandomForestClassifier(max_depth=5, n_jobs=10, oob_score=True, random_state=42)

https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

In [10]:
#generate predeictions
y_test_bow_preds = classifier_rf.predict(X_test_bow)

In [11]:
y_test_bow_preds

array([1, 1, 1, ..., 1, 1, 1])

In [12]:
print('Classification Report pe setul de test\n',
      classification_report(y_test_bow, y_test_bow_preds)
      )

Classification Report pe setul de test
               precision    recall  f1-score   support

           0       0.98      0.10      0.18       582
           1       0.71      1.00      0.83      1291

    accuracy                           0.72      1873
   macro avg       0.85      0.55      0.50      1873
weighted avg       0.79      0.72      0.63      1873



## Grid Search

In [18]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

params = {
    'max_depth': [5,10,20],
    'min_samples_leaf': [50,100,200],
    'n_estimators': [30,50,100]
}


# Instantiate the grid search model
grid_search = GridSearchCV(estimator=rf,
                           param_grid=params,
                           cv = 4,
                           n_jobs=4, verbose=1, scoring="accuracy")

grid_search.fit(X_train_bow, y_train_bow)

Fitting 4 folds for each of 27 candidates, totalling 108 fits


GridSearchCV(cv=4, estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
             n_jobs=4,
             param_grid={'max_depth': [5, 10, 20],
                         'min_samples_leaf': [50, 100, 200],
                         'n_estimators': [30, 50, 100]},
             scoring='accuracy', verbose=1)

In [19]:
grid_search.best_score_

np.float64(0.7426588360918314)

In [20]:
rf_best = grid_search.best_estimator_
rf_best

RandomForestClassifier(max_depth=20, min_samples_leaf=50, n_estimators=30,
                       n_jobs=-1, random_state=42)

In [21]:
#generate predeictions
y_test_bow_preds_grid = rf_best.predict(X_test_bow)

In [22]:
print('Classification Report pe setul de test\n',
      classification_report(y_test_bow, y_test_bow_preds_grid)
      )

Classification Report pe setul de test
               precision    recall  f1-score   support

           0       0.97      0.20      0.33       582
           1       0.73      1.00      0.85      1291

    accuracy                           0.75      1873
   macro avg       0.85      0.60      0.59      1873
weighted avg       0.81      0.75      0.68      1873

